# Modern Image Classification (2026): TensorFlow 2.21 + Keras 3

**Maintainer: Harpreet Singh**

A compact, modern companion lesson for the original TensorFlow LiveLessons notebooks. It demonstrates deterministic setup, a `tf.data` input pipeline, callbacks, TensorBoard logging, evaluation, and saving in the native `.keras` format.

> This notebook downloads CIFAR-10 from Keras on first run. Training time and accuracy depend on hardware and runtime; no accuracy is promised in advance.


## 1. Environment

Use Python 3.11 or 3.12 in a clean environment. Install packages using `requirements-modern.txt` from the repository root. GPU setup depends on your OS, driver, and supported TensorFlow wheel; see the official TensorFlow installation guide.

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import tensorflow as tf
import keras
from keras import layers

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)

print("Python TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("Visible GPUs:", tf.config.list_physical_devices("GPU"))


## 2. Load and prepare data

CIFAR-10 contains 32×32 colour images across 10 classes. Normalization is performed in the model so the preprocessing travels with the saved model.

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
y_train = y_train.squeeze().astype("int32")
y_test = y_test.squeeze().astype("int32")

BATCH_SIZE = 128
AUTOTUNE = tf.data.AUTOTUNE
train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(10_000, seed=SEED, reshuffle_each_iteration=True)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
test_ds = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
print(f"Train: {len(x_train):,} | Test: {len(x_test):,}")


## 3. Build a compact CNN

This is a baseline for learning, not a state-of-the-art benchmark. Data augmentation is active only during training.

In [ ]:
data_augmentation = keras.Sequential(
    [layers.RandomFlip("horizontal"), layers.RandomTranslation(0.08, 0.08)],
    name="data_augmentation",
)

inputs = keras.Input(shape=(32, 32, 3), name="image")
x = layers.Rescaling(1.0 / 255)(inputs)
x = data_augmentation(x)
for filters in (32, 64, 128):
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.15)(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(10, name="logits")(x)
model = keras.Model(inputs, outputs, name="harpreet_cifar10_cnn")
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
)
model.summary()


## 4. Train with callbacks

Early stopping restores the best validation weights. Checkpoints and TensorBoard logs are written to `artifacts/` (ignored by Git).

In [ ]:
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)
callbacks = [
    keras.callbacks.ModelCheckpoint(
        ARTIFACTS / "best_cifar10.keras", monitor="val_accuracy",
        mode="max", save_best_only=True, verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy", mode="max", patience=4, restore_best_weights=True,
    ),
    keras.callbacks.TensorBoard(log_dir=str(ARTIFACTS / "tensorboard")),
]
history = model.fit(
    train_ds, validation_data=test_ds, epochs=20, callbacks=callbacks,
)


## 5. Evaluate and save

The test set is used here for a simple teaching demo. For rigorous model selection, keep a separate validation set and evaluate on the test set only once at the end.

In [ ]:
test_metrics = model.evaluate(test_ds, return_dict=True, verbose=0)
print({name: round(float(value), 4) for name, value in test_metrics.items()})
model.save(ARTIFACTS / "final_cifar10.keras")

reloaded = keras.models.load_model(ARTIFACTS / "final_cifar10.keras")
print("Reload check:", reloaded.evaluate(test_ds, return_dict=True, verbose=0))


## 6. Predict a sample

The output layer contains logits; `softmax` converts them to class probabilities for display.

In [ ]:
CLASS_NAMES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]
logits = reloaded.predict(x_test[:8], verbose=0)
probabilities = tf.nn.softmax(logits, axis=-1).numpy()
for i, probs in enumerate(probabilities):
    pred = int(np.argmax(probs))
    print(f"sample={i} true={CLASS_NAMES[int(y_test[i])]} "
          f"pred={CLASS_NAMES[pred]} confidence={probs[pred]:.3f}")


## 7. Explore further

- Open TensorBoard: `tensorboard --logdir artifacts/tensorboard`
- Compare augmentation on/off and document the same-seed results.
- Add precision/recall, a confusion matrix, or calibration analysis.
- For deployment, evaluate latency and export to a target format only after checking operator support.

**Reproducibility note:** seeding improves repeatability but does not guarantee bit-for-bit determinism across hardware, drivers, and kernels.